In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("dataset_for_dano_fuel.csv", parse_dates=["week"])

df["week"].min(), df["week"].max()

In [ ]:
df.info()

In [ ]:
df["week"].dt.to_period("W").nunique()

In [ ]:
df["week"].dt.to_period("W").value_counts().sort_index().plot(kind="bar", figsize=(15,4))

In [ ]:
weeks_per_user = df.groupby("party_rk")["week"].nunique()
weeks_per_user.describe()

In [ ]:
weeks_per_user.hist(bins=30, figsize=(6,4))

In [ ]:
fee_change_per_user = df.groupby("party_rk")["service_fee_amt"].nunique()
fee_change_per_user.describe()

In [ ]:
fee_change_per_user.value_counts()

In [ ]:
fee_transitions = fee_change_per_user[fee_change_per_user >= 2].count()
total_users = df["party_rk"].nunique()

fee_transitions / total_users

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# только пользователи с 2+ уникальными СС
valid_users = fee_change_per_user[fee_change_per_user >= 2].index
df_valid = df[df['party_rk'].isin(valid_users)]

# Pivot для heatmap: строки = user, столбцы = week, значения = service_fee_amt
pivot = df_valid.pivot_table(
    index='party_rk',
    columns='week',
    values='service_fee_amt',
    aggfunc='mean'
)

plt.figure(figsize=(15,10))
sns.heatmap(pivot, cmap='viridis', cbar_kws={'label':'Service Fee'})
plt.title('Changes in Service Fee per User Over Weeks')
plt.xlabel('Week')
plt.ylabel('User ID')
plt.show()


In [ ]:
# считаем количество заправок на пользователя и неделю
freq = df_valid.groupby(['party_rk', 'week'])['orders_cnt'].sum().reset_index()

# добавляем средний СС на неделю
fee = df_valid.groupby(['party_rk', 'week'])['service_fee_amt'].mean().reset_index()
merged = freq.merge(fee, on=['party_rk','week'])

# scatter plot: частота vs СС
plt.figure(figsize=(8,5))
plt.scatter(merged['service_fee_amt'], merged['orders_cnt'], alpha=0.3)
plt.xlabel('Service Fee')
plt.ylabel('Orders Count')
plt.title('Frequency of Refills vs Service Fee')
plt.show()


In [ ]:
df['week'].head()

In [ ]:
last_week = df.groupby("party_rk")["week"].max()
first_week = df.groupby("party_rk")["week"].min()
active_span = (last_week - first_week).dt.days

plt.figure(figsize=(6,4))
active_span.hist(bins=30)

plt.title("Распределение длительности активности пользователей (в днях)")
plt.xlabel("Длительность активности, дней")
plt.ylabel("Количество пользователей")

plt.tight_layout()
plt.show()

In [ ]:
trips = df.groupby("party_rk").size()
trips.hist(bins=50, figsize=(6,4))

In [ ]:
plt.scatter(active_span, trips, alpha=0.1)
plt.xlabel("Длительность активности, дней")
plt.ylabel("Количество заправок")

In [ ]:
df_sorted = df.sort_values(["party_rk", "week"])

df_sorted["prev_fee"] = df_sorted.groupby("party_rk")["service_fee_amt"].shift(1)
df_sorted["prev_orders"] = df_sorted.groupby("party_rk")["orders_cnt"].shift(1)
df_sorted["prev_liters"] = df_sorted.groupby("party_rk")["liters"].shift(1)
df_sorted["prev_gmv"] = df_sorted.groupby("party_rk")["gmv"].shift(1)

df_sorted["fee_delta"] = df_sorted["service_fee_amt"] - df_sorted["prev_fee"]
df_sorted["orders_delta"] = df_sorted["orders_cnt"] - df_sorted["prev_orders"]
df_sorted["liters_delta"] = df_sorted["liters"] - df_sorted["prev_liters"]
df_sorted["gmv_delta"] = df_sorted["gmv"] - df_sorted["prev_gmv"]

In [ ]:
df_sorted[["fee_delta", "orders_delta", "liters_delta", "gmv_delta"]].corr()

In [ ]:
plt.figure(figsize=(6,4))
plt.scatter(df_sorted["fee_delta"], df_sorted["orders_delta"], alpha=0.1)
plt.xlabel("Изменение сервисного сбора")
plt.ylabel("Изменение количества заказов")
plt.title("Как изменение СС влияет на количество заказов")
plt.show()

In [ ]:
df.boxplot(column="orders_cnt", by="service_fee_amt", figsize=(10,6))
plt.title("Распределение заказов по значениям сервисного сбора")
plt.suptitle("")
plt.xlabel("Сервисный сбор")
plt.ylabel("Количество заказов")

In [ ]:
import scipy.stats as stats

grp19 = df[df["service_fee_amt"] == 19]["orders_cnt"]
grp39 = df[df["service_fee_amt"] == 39]["orders_cnt"]

stats.ttest_ind(grp19, grp39, equal_var=False)

In [ ]:
import statsmodels.formula.api as smf
import numpy as np

df = df.copy()

# удалить полностью пустые строки
df = df.replace("", np.nan)

# удалить строки с NA в важных признаках
df = df.dropna(subset=["service_fee_amt", "orders_cnt", "age", "liters", "entries_cnt"])

# ограничить количество уникальных категорий
# например region заменить на 20 топовых, остальные = "OTHER"
top_regions = df["region"].value_counts().nlargest(20).index
df["region"] = df["region"].where(df["region"].isin(top_regions), "OTHER")

# job_title вообще лучше убрать — слишком много уникальных значений
del df["job_title"]

model = smf.ols(
    formula="orders_cnt ~ service_fee_amt + age + liters + entries_cnt + C(platform) + C(region)",
    data=df
).fit()

print(model.summary())


In [ ]:
df["week"] = pd.to_datetime(df["week"])

# создаём номер недели в отсчёте от минимума
week_order = {w: i for i, w in enumerate(sorted(df["week"].unique()))}
df["week_idx"] = df["week"].map(week_order)

last_week_idx = df.groupby("party_rk")["week_idx"].max()
max_week_idx = df["week_idx"].max()

churn = (last_week_idx < max_week_idx)

df_users = last_week_idx.to_frame("last_week_idx")
df_users["churn"] = (df_users["last_week_idx"] < max_week_idx).astype(int)

# подмердживаем демографию и средние fee
user_features = df.groupby("party_rk").agg({
    "service_fee_amt": "mean",
    "age": "mean",
    "entries_cnt": "sum",
    "orders_cnt": "sum",
    "bundle_name": lambda x: x.mode().iloc[0] if len(x.mode()) else "Unknown"
})

df_model = df_users.join(user_features)

import statsmodels.formula.api as smf

model_churn = smf.logit(
    formula="churn ~ service_fee_amt + age + entries_cnt + orders_cnt + C(bundle_name)",
    data=df_model
).fit()

print(model_churn.summary())

In [ ]:
df.info()

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

df = df.copy()

# Заменяем пустые строки на NaN
df = df.replace("", np.nan)

# Числовые колонки: age, children_cnt — заполняем медианой
for col in ["age", "children_cnt"]:
    df[col] = df[col].fillna(df[col].median())

# Категориальные колонки — заполняем Unknown
for col in ["gender_cd", "education_level_cd", "marital_status_cd", "region", "bundle_name"]:
    df[col] = df[col].fillna("Unknown")

# Ограничиваем количество категорий, например, топ-20 регионов
top_regions = df["region"].value_counts().nlargest(20).index
df["region"] = df["region"].where(df["region"].isin(top_regions), "OTHER")

# Убираем job_title, слишком много уникальных значений
if "job_title" in df.columns:
    del df["job_title"]

# OLS-модель
model = smf.ols(
    formula="orders_cnt ~ service_fee_amt + age + liters + entries_cnt + C(platform) + C(region)",
    data=df
).fit()

print(model.summary())

In [ ]:
# --- Churn-модель ---
df["week"] = pd.to_datetime(df["week"])
week_order = {w: i for i, w in enumerate(sorted(df["week"].unique()))}
df["week_idx"] = df["week"].map(week_order)

last_week_idx = df.groupby("party_rk")["week_idx"].max()
max_week_idx = df["week_idx"].max()

df_users = last_week_idx.to_frame("last_week_idx")
df_users["churn"] = (df_users["last_week_idx"] < max_week_idx).astype(int)

# Средние показатели по пользователям
user_features = df.groupby("party_rk").agg({
    "service_fee_amt": "mean",
    "age": "mean",
    "entries_cnt": "sum",
    "orders_cnt": "sum",
    "bundle_name": lambda x: x.mode().iloc[0] if len(x.mode()) else "Unknown"
})

df_model = df_users.join(user_features)

# Логистическая регрессия
model_churn = smf.logit(
    formula="churn ~ service_fee_amt + age + entries_cnt + orders_cnt + C(bundle_name)",
    data=df_model
).fit()

print(model_churn.summary())

In [ ]:
df["week"] = pd.to_datetime(df["week"])

fee_by_week = (
    pd.crosstab(
        df["week"],
        df["service_fee_amt"],
        normalize="index"
    )
    .fillna(0)
)

fee_by_week

In [ ]:
from scipy.stats import entropy

entropy_by_week = fee_by_week.apply(
    lambda row: entropy(row + 1e-9),
    axis=1
)

entropy_by_week


In [ ]:
df = pd.read_csv("dataset_for_dano_fuel.csv")

df["week"] = pd.to_datetime(df["week"])

weeks_sorted = sorted(df["week"].unique())
print(weeks_sorted)

df["week"] = df["week"].dt.isocalendar().week

weeks_sorted = sorted(df["week"].unique())

first_week = weeks_sorted[0]
last_4_weeks = weeks_sorted[-4:]

outlier_mask = (df["service_fee_amt"] == 69) & (
    (df["week"] == first_week) | (df["week"].isin(last_4_weeks))
)

print(f"Удаляем выбросов: {outlier_mask.sum()} строк")
df = df.loc[~outlier_mask].copy()

df.drop(columns=["region"], inplace=True)

print("Итоговые недели:", sorted(df["week"].unique()))

df.to_csv("dataset_for_dano_fuel_clean.csv", index=False)